# LLaVA-style LVLM Training on Google Colab

**Objective**: Train a Large Vision-Language Model combining:
- Frozen CLIP ViT-B/32 visual encoder
- Trainable 3-layer MLP projection (512 → 4096)
- LoRA-finetuned LLaMA-2-7B (r=16, alpha=32)

**Key Features**:
- Mixed precision (float16) training with GradScaler
- Cosine scheduler with warmup
- LLaVA-format dataset (image + conversations)
- Loss computed only on answer tokens (prompt/visual masked with -100)
- Gradient checkpointing for memory efficiency

**Timeline**: 3 epochs on 100-1000 samples ≈ 2-4 hours on T4 GPU

---

## Setup & Configuration

## 1. Environment Setup & Dependencies

In [ ]:
# Check GPU availability and setup
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"CUDA version: {torch.version.cuda}")


In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    "transformers>=4.35.0",
    "peft>=0.7.0",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "Pillow>=9.0.0",
    "git+https://github.com/openai/CLIP.git",
    "huggingface-hub>=0.19.0",
    "tqdm>=4.65.0",
]

print("Installing required packages...")
for pkg in packages:
    print(f"  Installing {pkg.split('/')[-1].split('@')[0]}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✓ All packages installed successfully")


## 2. HuggingFace Authentication

In [ ]:
from huggingface_hub import login
import getpass

# Option 1: Use token from Colab Secret (recommended)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✓ Authenticated with HuggingFace (Colab Secret)")
except:
    pass

# Option 2: Enter token manually
if not hf_token:
    print("Enter your HuggingFace token (get it from https://huggingface.co/settings/tokens):")
    hf_token = getpass.getpass("Token: ")
    login(token=hf_token)
    print("✓ Authenticated with HuggingFace")

# Verify by loading tokenizer
from transformers import AutoTokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
    print("✓ LLaMA-2-7B tokenizer loaded successfully")
except Exception as e:
    print(f"✗ Failed to load tokenizer: {e}")


## 3. Import Core Modules

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from PIL import Image
import clip
import logging
from tqdm import tqdm
from datetime import datetime
import numpy as np

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ All imports successful")


## 4. Define LVLM Components

### 4.1 Frozen CLIP Encoder

In [ ]:
class FrozenCLIPEncoder(nn.Module):
    """Frozen CLIP visual encoder outputting L2-normalized embeddings."""
    
    EMBED_DIM = {"ViT-B/32": 512, "ViT-B/16": 512, "ViT-L/14": 768}
    
    def __init__(self, model_name: str = "ViT-B/32", device: str = "cuda"):
        super().__init__()
        model, self.preprocess = clip.load(model_name, device=device)
        self.visual = model.visual
        self.embed_dim = self.EMBED_DIM.get(model_name, 512)
        self.device = device
        
        # Freeze all parameters
        for param in self.parameters():
            param.requires_grad = False
        
        logger.info(f"Loaded Frozen CLIP {model_name} (embed_dim={self.embed_dim})")
    
    @torch.no_grad()
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pixel_values: (B, 3, H, W) CLIP-preprocessed images
        Returns:
            embeddings: (B, embed_dim) L2-normalized
        """
        pixel_values = pixel_values.to(self.device)
        feats = self.visual(pixel_values)  # (B, embed_dim)
        return feats / feats.norm(dim=-1, keepdim=True)

print("✓ FrozenCLIPEncoder defined")


In [ ]:
class VisualProjection(nn.Module):
    """3-layer MLP projection mapping CLIP embeddings to visual tokens."""
    
    def __init__(
        self,
        clip_dim: int = 512,
        llm_dim: int = 4096,
        num_tokens: int = 32,
        hidden_dim: int | None = None,
        dropout: float = 0.0,
    ):
        super().__init__()
        hidden_dim = hidden_dim or llm_dim
        out_dim = num_tokens * llm_dim
        
        self.num_tokens = num_tokens
        self.llm_dim = llm_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(clip_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )
    
    def forward(self, clip_embeds: torch.Tensor) -> torch.Tensor:
        """
        Args:
            clip_embeds: (B, clip_dim)
        Returns:
            visual_tokens: (B, num_tokens, llm_dim)
        """
        out = self.mlp(clip_embeds)  # (B, num_tokens * llm_dim)
        return out.view(-1, self.num_tokens, self.llm_dim)

print("✓ VisualProjection defined")


In [ ]:
def build_lora_llama(
    model_name_or_path: str = "meta-llama/Llama-2-7b-hf",
    lora_r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.05,
    target_modules: list | None = None,
    torch_dtype: torch.dtype = torch.float16,
):
    """Load LLaMA-2-7B and wrap with LoRA adapters."""
    
    # Load tokenizer
    logger.info(f"Loading tokenizer from {model_name_or_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load base model with specified dtype
    logger.info(f"Loading model in {torch_dtype}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    
    # Configure LoRA
    if target_modules is None:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
    
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias="none",
    )
    
    # Wrap with LoRA
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    
    return model, tokenizer

print("✓ build_lora_llama defined")


In [ ]:
class LVLM(nn.Module):
    """Large Vision-Language Model combining CLIP, projection, and LoRA-LLaMA."""
    
    def __init__(
        self,
        llama_model: nn.Module,
        tokenizer,
        clip_model_name: str = "ViT-B/32",
        num_visual_tokens: int = 32,
        llm_dim: int = 4096,
        device: str = "cuda",
    ):
        super().__init__()
        self.device = device
        self.num_visual_tokens = num_visual_tokens
        
        self.clip = FrozenCLIPEncoder(model_name=clip_model_name, device=device)
        self.projection = VisualProjection(
            clip_dim=self.clip.embed_dim,
            llm_dim=llm_dim,
            num_tokens=num_visual_tokens,
        )
        self.llama = llama_model
        self.tokenizer = tokenizer
    
    def _get_model_dtype(self) -> torch.dtype:
        return next(self.llama.parameters()).dtype
    
    def _get_llama_embeddings(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.llama.get_input_embeddings()(input_ids)
    
    def _extract_visual_tokens(self, pixel_values: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            clip_embeds = self.clip(pixel_values)
        model_dtype = self._get_model_dtype()
        clip_embeds = clip_embeds.to(dtype=model_dtype)
        return self.projection(clip_embeds)
    
    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        labels: torch.Tensor | None = None,
    ):
        B = pixel_values.size(0)
        
        # Concatenate visual and text tokens
        visual_tokens = self._extract_visual_tokens(pixel_values)
        text_embeds = self._get_llama_embeddings(input_ids)
        inputs_embeds = torch.cat([visual_tokens, text_embeds], dim=1)
        
        # Build attention mask
        vis_mask = torch.ones(B, self.num_visual_tokens, device=input_ids.device, dtype=torch.long)
        txt_mask = (input_ids != self.tokenizer.pad_token_id).long()
        attention_mask = torch.cat([vis_mask, txt_mask], dim=1)
        
        return self.llama(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True,
        )

print("✓ LVLM defined")


### 4.2 Dataset & Collator

In [ ]:
class LLaVADataset(Dataset):
    """LLaVA-format dataset for instruction tuning."""
    
    PROMPT_TEMPLATE = "Question: {question}\nAnswer:"
    
    def __init__(
        self,
        samples: list,
        image_dir: str,
        tokenizer,
        clip_preprocess,
        num_visual_tokens: int = 32,
        max_length: int = 512,
    ):
        self.samples = samples
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.clip_preprocess = clip_preprocess
        self.num_visual_tokens = num_visual_tokens
        self.max_length = max_length
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx: int):
        item = self.samples[idx]
        
        # Extract Q&A from conversations
        convs = item.get("conversations", [])
        question = next((c["value"] for c in convs if c["from"] == "human"), "").strip()
        answer = next((c["value"] for c in convs if c["from"] == "gpt"), "").strip()
        
        # Load image
        img_path = os.path.join(self.image_dir, item["image"])
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.clip_preprocess(image)
        
        # Tokenize
        prompt = self.PROMPT_TEMPLATE.format(question=question)
        full_text = prompt + " " + answer + self.tokenizer.eos_token
        
        enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        
        # Compute prompt length for masking
        prompt_len = self.tokenizer(prompt, return_tensors="pt")["input_ids"].shape[-1]
        
        # Create labels
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        
        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "labels": labels,
        }


def llava_collate_fn(batch, num_visual_tokens):
    """Collate batch and prepend visual token masks."""
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    input_ids = torch.stack([b["input_ids"] for b in batch])
    text_labels = torch.stack([b["labels"] for b in batch])
    
    # Prepend visual token mask
    B = pixel_values.size(0)
    visual_mask = torch.full((B, num_visual_tokens), -100, dtype=torch.long)
    labels = torch.cat([visual_mask, text_labels], dim=1)
    
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "labels": labels,
    }

print("✓ LLaVADataset and collate_fn defined")


## 5. Create Dummy Data for Testing

In [ ]:
# Create dummy data
os.makedirs("./data/images", exist_ok=True)

# Generate dummy images
print("Creating 16 dummy images...")
for i in range(16):
    # Create random RGB image
    image_array = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
    img = Image.fromarray(image_array)
    img.save(f"./data/images/image_{i:03d}.jpg")

# Create dummy dataset JSON
dummy_samples = [
    {
        "image": f"image_{i:03d}.jpg",
        "conversations": [
            {
                "from": "human",
                "value": "What can you see in this image?",
            },
            {
                "from": "gpt",
                "value": "This is an image with random colored pixels showing various colors and patterns.",
            },
        ],
    }
    for i in range(16)
]

with open("./data/train_data.json", "w") as f:
    json.dump(dummy_samples, f)

print(f"✓ Created {len(dummy_samples)} dummy samples")
print(f"✓ Dataset JSON saved to ./data/train_data.json")


## 6. Training Configuration

In [ ]:
# Training configuration
cfg = {
    # Model config
    "llama_model_name": "meta-llama/Llama-2-7b-hf",
    "clip_model_name": "ViT-B/32",
    "num_visual_tokens": 32,
    "llm_dim": 4096,
    
    # LoRA config
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    
    # Training config
    "batch_size": 2,
    "num_epochs": 1,  # Start with 1 epoch for testing
    "learning_rate": 2e-4,
    "warmup_steps": 10,
    "max_grad_norm": 1.0,
    "gradient_accumulation_steps": 1,
    
    # Data config
    "data_path": "./data/train_data.json",
    "image_dir": "./data/images",
    "output_dir": "./checkpoints",
    "max_length": 512,
    
    # Device
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "mixed_precision": True,
    "seed": 42,
}

print("Training Configuration:")
for key, value in cfg.items():
    print(f"  {key}: {value}")

torch.manual_seed(cfg["seed"])
if cfg["device"] == "cuda":
    torch.cuda.manual_seed_all(cfg["seed"])


## 7. Build Model

In [ ]:
print("Building LVLM model...")

# Build LoRA-LLaMA
llama_model, tokenizer = build_lora_llama(
    model_name_or_path=cfg["llama_model_name"],
    lora_r=cfg["lora_r"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=cfg["lora_dropout"],
    torch_dtype=torch.float16 if cfg["mixed_precision"] else torch.float32,
)

# Build LVLM
lvlm = LVLM(
    llama_model=llama_model,
    tokenizer=tokenizer,
    clip_model_name=cfg["clip_model_name"],
    num_visual_tokens=cfg["num_visual_tokens"],
    llm_dim=cfg["llm_dim"],
    device=cfg["device"],
)

# Count parameters
total_params = sum(p.numel() for p in lvlm.parameters())
trainable_params = sum(p.numel() for p in lvlm.parameters() if p.requires_grad)

print(f"✓ LVLM built successfully")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable ratio: {100.*trainable_params/total_params:.2f}%")


## 8. Load Dataset

In [ ]:
print("Loading dataset...")

# Load samples
with open(cfg["data_path"]) as f:
    samples = json.load(f)

print(f"✓ Loaded {len(samples)} samples")

# Create dataset
dataset = LLaVADataset(
    samples=samples,
    image_dir=cfg["image_dir"],
    tokenizer=tokenizer,
    clip_preprocess=lvlm.clip.preprocess,
    num_visual_tokens=cfg["num_visual_tokens"],
    max_length=cfg["max_length"],
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=cfg["batch_size"],
    shuffle=True,
    num_workers=0,  # Colab compatibility
    collate_fn=lambda b: llava_collate_fn(b, cfg["num_visual_tokens"]),
)

print(f"✓ DataLoader created with batch_size={cfg['batch_size']}")
print(f"  Total batches per epoch: {len(dataloader)}")


## 9. Training Loop with Mixed Precision

In [ ]:
print("Initializing training components...")

# Move model to device
lvlm = lvlm.to(cfg["device"])
lvlm.train()

# Optimizer
trainable_params = filter(lambda p: p.requires_grad, lvlm.parameters())
optimizer = AdamW(trainable_params, lr=cfg["learning_rate"])

# Scheduler
total_steps = len(dataloader) * cfg["num_epochs"]
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg["warmup_steps"],
    num_training_steps=total_steps,
)

# Mixed precision
scaler = GradScaler() if cfg["mixed_precision"] else None

print(f"✓ Optimizer: AdamW (lr={cfg['learning_rate']})")
print(f"✓ Scheduler: Cosine with warmup ({cfg['warmup_steps']} steps)")
print(f"✓ Mixed precision: {cfg['mixed_precision']}")
print(f"✓ Total training steps: {total_steps}")

# Training loop
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

global_step = 0
best_loss = float('inf')

for epoch in range(cfg["num_epochs"]):
    epoch_loss = 0.0
    pbar = tqdm(
        dataloader,
        desc=f"Epoch {epoch+1}/{cfg['num_epochs']}",
        total=len(dataloader),
    )
    
    for batch_idx, batch in enumerate(pbar):
        # Move batch to device
        pixel_values = batch["pixel_values"].to(cfg["device"])
        input_ids = batch["input_ids"].to(cfg["device"])
        labels = batch["labels"].to(cfg["device"])
        
        # Forward pass with mixed precision
        if cfg["mixed_precision"]:
            with autocast(dtype=torch.float16):
                outputs = lvlm(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    labels=labels,
                )
                loss = outputs.loss / cfg["gradient_accumulation_steps"]
            
            # Backward with scaler
            scaler.scale(loss).backward()
        else:
            outputs = lvlm(
                pixel_values=pixel_values,
                input_ids=input_ids,
                labels=labels,
            )
            loss = outputs.loss / cfg["gradient_accumulation_steps"]
            loss.backward()
        
        epoch_loss += loss.item() * cfg["gradient_accumulation_steps"]
        
        # Gradient accumulation step
        if (batch_idx + 1) % cfg["gradient_accumulation_steps"] == 0:
            if cfg["mixed_precision"]:
                scaler.unscale_(optimizer)
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(lvlm.parameters(), cfg["max_grad_norm"])
            
            # Optimizer step
            if cfg["mixed_precision"]:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1
            
            # Update progress bar
            pbar.set_postfix({
                'loss': loss.item() * cfg["gradient_accumulation_steps"],
                'lr': optimizer.param_groups[0]['lr'],
            })
    
    # Epoch summary
    avg_epoch_loss = epoch_loss / len(dataloader)
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Average Loss: {avg_epoch_loss:.4f}")
    print(f"  Global Steps: {global_step}")
    
    # Save checkpoint
    os.makedirs(cfg["output_dir"], exist_ok=True)
    checkpoint_path = os.path.join(cfg["output_dir"], f"epoch_{epoch+1}.pt")
    torch.save(lvlm.state_dict(), checkpoint_path)
    print(f"  Checkpoint saved to {checkpoint_path}")

print("\n" + "="*80)
print("✓ TRAINING COMPLETE")
print("="*80)


## 10. Inference Example

In [ ]:
# Test inference on a sample
print("Testing inference...")

# Load a sample image
test_image_path = "./data/images/image_000.jpg"
test_question = "What can you see in this image?"

# Preprocess image
test_image = Image.open(test_image_path).convert("RGB")
pixel_values = lvlm.clip.preprocess(test_image).unsqueeze(0).to(cfg["device"])

# Tokenize question
prompt = f"Question: {test_question}\nAnswer:"
input_ids = tokenizer(
    prompt,
    return_tensors="pt",
    max_length=512,
    padding="max_length",
    truncation=True,
)["input_ids"].to(cfg["device"])

print(f"Input image: {test_image_path}")
print(f"Question: {test_question}")
print("\nGenerating response...")

# Generate response
with torch.no_grad():
    outputs = lvlm.generate(
        pixel_values=pixel_values,
        input_ids=input_ids,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
    )

# Decode
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nModel Response:\n{response}")


## 11. Next Steps for Real Training

### On Google Colab:
1. **Upload real LLaVA dataset**: Replace `./data/train_data.json` with your actual LLaVA-format data
2. **Increase training epochs**: Change `cfg["num_epochs"]` from 1 to 3+ for better convergence
3. **Adjust batch size**: Try batch_size=8 if GPU memory allows (check `torch.cuda.mem_get_info()`)
4. **Enable checkpointing**: Add `gradient_checkpointing=True` to reduce memory usage
5. **Monitor metrics**: Track loss, throughput, and GPU memory

### Memory Management:
- Current model: ~7B params (LLaMA) + LoRA (0.3%) + projection (~2.3M)
- Frozen CLIP: 0 gradients
- Expected memory: ~13-16 GB on single GPU

### Performance Tips:
- Use `pin_memory=True` in DataLoader for faster I/O
- Enable gradient accumulation for effective larger batch sizes
- Save checkpoints every N steps, not just epochs
- Use mixed precision (already enabled) for 2x speedup

### For Research Paper:
- Save loss curves and metrics
- Document hyperparameters used
- Compare with baseline LLaVA results
- Include qualitative examples of model predictions